# Arya LoRA (optional, free Colab T4)

English + Persian comments. **Do not run training cells on a laptop.** Dataset cells are dry-run safe.

Base: `unsloth/Qwen3-1.7B-Instruct` 4-bit. Export GGUF Q4_K_M and install via the app custom-URL slot.

## 1. Dataset build (dry-run OK — no GPU)

In [ ]:
# Dry-run: synthesize ChatML samples from the bench file + compiler patterns.
import json, csv, os, random
from pathlib import Path

BENCH = Path("app/src/main/assets/benchmarks/arya_bench_fa.json")
PATTERNS = Path("training/patterns.tsv")  # optional export from compiler tests

samples = []
if BENCH.exists():
    data = json.loads(BENCH.read_text(encoding="utf-8"))
    items = data if isinstance(data, list) else data.get("items", data.get("cases", []))
    for row in items:
        user = row.get("input") or row.get("prompt") or row.get("task") or ""
        tool = row.get("tool") or row.get("expected_tool")
        args = row.get("args") or row.get("expected_args") or {}
        if not user:
            continue
        if tool:
            assistant = f'<tool_call>{{"name":"{tool}","arguments":{json.dumps(args, ensure_ascii=False)}}}</tool_call>'
        else:
            assistant = row.get("answer") or "متوجه شدم. لطفاً کمی بیشتر توضیح بده."
        samples.append({"user": user, "assistant": assistant})

# Negative / chit-chat samples
samples.append({"user": "سلام خوبی؟", "assistant": "سلام! خوبم، چطور می‌تونم کمکت کنم؟"})
samples.append({"user": "hi", "assistant": "Hi! How can I help?"})

seen = set()
deduped = []
for s in samples:
    key = s["user"].strip()
    if key and key not in seen:
        seen.add(key)
        deduped.append(s)
random.Random(0).shuffle(deduped)
n = len(deduped)
split = max(1, int(n * 0.9))
train, val = deduped[:split], deduped[split:]
print(f"samples={n} train={len(train)} val={len(val)}")
Path("training").mkdir(exist_ok=True)
Path("training/arya_sft_train.jsonl").write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in train), encoding="utf-8")
Path("training/arya_sft_val.jsonl").write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in val), encoding="utf-8")

## 2. Training — RUN ON COLAB (T4)
QLoRA r=16 alpha=32 lr 2e-4, 1–3 epochs, packing on.

In [ ]:
# RUN ON COLAB — requires GPU + unsloth
# !pip install -q unsloth
print("Skip locally. On Colab: load unsloth/Qwen3-1.7B-Instruct 4-bit, attach LoRA r=16, train on arya_sft_train.jsonl")

## 3. Export GGUF Q4_K_M + Drive

In [ ]:
# RUN ON COLAB after merge
print("Merge adapter → convert to GGUF Q4_K_M via Unsloth helper / llama.cpp convert")
print("Upload the .gguf to Drive, then paste the public URL into Arya Settings → Custom local model URL")

## 4. Sanity eval (optional, Colab)

In [ ]:
# Compare tool-call exact-match on bench prompts vs base model.
print("Print exact-match rate vs base. Target is directional, not a ship gate.")